# First SFT: Code Bug Detection

Fine-tune a model to learn the checked-in code bug detection examples.

This notebook is intentionally limited to supervised fine-tuning:

1. Validate the exact checked-in training and validation files.
2. Authenticate with `DefaultAzureCredential` and create the OpenAI client through `AIProjectClient.get_openai_client()`.
3. Upload the files and submit one `OpenAI-OSS gpt-oss-120b` version `1` `GlobalStandard` job.
4. Monitor the job to a terminal state.
5. Download service result CSVs and summarize only validation loss and validation token accuracy.

The workflow stops after service training metrics and makes no claim about model behavior after training.

**Reference only:** A prior demonstration run reported validation loss 1.62558 -> 1.33327. It is context only; this notebook submits and measures a fresh job.

**Unused provenance:** The source demo also tracked a 10-row held-out scale. Those rows are not copied, uploaded, or used here.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.


## 1. Configure the training recipe

Copy this module's `.env.template` to a module-local `.env`, set your own `FOUNDRY_PROJECT_ENDPOINT`, and sign in with `az login`. The checked-in model, version, training type, hyperparameters, and hash-pinned curriculum data remain fixed and visible below.

Run the preparation cells to validate the data, then execute the client and training cells to authenticate, upload, and submit or resume a paid job. Your project must support the selected model and training type.


In [ ]:
import csv
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path


def find_cookbook_dir():
    current = Path.cwd().resolve()
    if (current / "data").is_dir() and (current / "notebooks").is_dir():
        return current
    if current.name == "notebooks" and (current.parent / "data").is_dir():
        return current.parent
    raise RuntimeError("Run this notebook from its cookbook directory or notebooks directory.")


COOKBOOK_DIR = find_cookbook_dir()
DATA_DIR = COOKBOOK_DIR / "data"
OUTPUT_DIR = COOKBOOK_DIR / "outputs"


from dotenv import load_dotenv

load_dotenv(COOKBOOK_DIR / ".env", override=False)


PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()

REGION = "West US 3"
MODEL = "gpt-oss-120b"
MODEL_VERSION = "1"
TRAINING_TYPE = "GlobalStandard"
STATE_PATH = OUTPUT_DIR / "submission-state.json"

TRAINING_PATH = DATA_DIR / "bug-detection-train.jsonl"
VALIDATION_PATH = DATA_DIR / "bug-detection-validation.jsonl"
EXPECTED = {
    "training": {
        "rows": 224,
        "bytes": 213931,
        "sha256": "5f0514a7ae2f529e0f78f7ba2a00beb54c525b16832030e616894025dbe3f1df"
    },
    "validation": {
        "rows": 20,
        "bytes": 20895,
        "sha256": "dd89ca979020003573bbf0b199b93fa712168ba56d953881648e90cff41e924c"
    }
}
HYPERPARAMETERS = {"n_epochs": 2, "learning_rate_multiplier": 0.8}


## 2. Validate the exact checked-in files

The shared message-schema, secret, duplicate, and split-isolation checks run first. Row count, canonical byte count, and SHA-256 must then match before upload.


In [ ]:
def canonical_bytes(path):
    return path.read_bytes().replace(b"\r\n", b"\n")


def sha256_file(path):
    return hashlib.sha256(canonical_bytes(path)).hexdigest()


from shared.dataset_validation import validate_jsonl, validate_split_isolation


def dataset_report(path, expected):
    report = validate_jsonl(path).require_valid()
    evidence = {
        "path": str(path.resolve()),
        "rows": report.rows,
        "bytes": len(canonical_bytes(path)),
        "sha256": sha256_file(path),
        "roles": report.roles,
        "duplicate_rows": report.duplicate_rows,
    }
    for key in ("rows", "bytes", "sha256"):
        if evidence[key] != expected[key]:
            raise ValueError(
                f"{path.name} {key} mismatch: expected {expected[key]!r}, found {evidence[key]!r}"
            )
    return evidence


validate_split_isolation({"training": TRAINING_PATH, "validation": VALIDATION_PATH})
dataset_evidence = {
    "source": "canonical",
    "training": dataset_report(TRAINING_PATH, EXPECTED["training"]),
    "validation": dataset_report(VALIDATION_PATH, EXPECTED["validation"]),
}
print(json.dumps(dataset_evidence, indent=2))


## 3. Authenticate

`DefaultAzureCredential` supplies Entra ID authentication. `AIProjectClient.get_openai_client()` supplies the OpenAI-compatible child client.


In [ ]:
credential = project_client = openai_client = None
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in your .env or process environment.")
credential = DefaultAzureCredential(process_timeout=120)
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"Recipe: {MODEL} version {MODEL_VERSION}, {TRAINING_TYPE}, {HYPERPARAMETERS}")


## 4. Upload and submit once

Executing this cell uploads data and submits or resumes a paid job. An ignored `outputs/submission-state.json` records the full recipe and input hashes; rerunning resumes the recorded job rather than submitting again. A different recipe or project is rejected until you intentionally archive that state. Job-creation POSTs have SDK retries disabled. Do not automatically retry an ambiguous job-creation failure: reconcile the submission in Foundry first.


In [ ]:
fine_tune_job = None
equivalence = {
    "project_endpoint": PROJECT_ENDPOINT,
    "region": REGION,
    "model": MODEL,
    "model_version": MODEL_VERSION,
    "training_type": TRAINING_TYPE,
    "hyperparameters": HYPERPARAMETERS,
    "dataset_sha256": {
        "training": dataset_evidence["training"]["sha256"],
        "validation": dataset_evidence["validation"]["sha256"],
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
state = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else None
if state and state.get("equivalence") != equivalence:
    raise RuntimeError("Saved state belongs to another recipe or project. Archive it before intentionally starting a new run.")
if state and state.get("job_id"):
    fine_tune_job = openai_client.fine_tuning.jobs.retrieve(state["job_id"])
    training_file_id = state["training_file_id"]
    validation_file_id = state["validation_file_id"]
    print(f"Resuming this recipe's job: {fine_tune_job.id}")
else:
    with TRAINING_PATH.open("rb") as source:
        training_file = openai_client.files.create(
            file=(TRAINING_PATH.name, canonical_bytes(TRAINING_PATH), "application/jsonl"), purpose="fine-tune", extra_headers={"Accept-Encoding": "identity"}
        )
    with VALIDATION_PATH.open("rb") as source:
        validation_file = openai_client.files.create(
            file=(VALIDATION_PATH.name, canonical_bytes(VALIDATION_PATH), "application/jsonl"), purpose="fine-tune", extra_headers={"Accept-Encoding": "identity"}
        )

    for uploaded_file in (training_file, validation_file):
        deadline = time.monotonic() + 600
        while True:
            uploaded = openai_client.files.retrieve(
                uploaded_file.id, extra_headers={"Accept-Encoding": "identity"}
            )
            if uploaded.status == "processed":
                break
            if uploaded.status in {"error", "failed", "expired", "cancelled"}:
                raise RuntimeError(f"File {uploaded_file.id} failed processing: {uploaded.status_details}")
            if time.monotonic() >= deadline:
                raise TimeoutError(f"File {uploaded_file.id} exceeded 600 seconds")
            time.sleep(10)

    training_file_id = training_file.id
    validation_file_id = validation_file.id
    method = {"type": "supervised", "supervised": {"hyperparameters": HYPERPARAMETERS}}
    fine_tune_job = openai_client.with_options(max_retries=0).fine_tuning.jobs.create(
        model=MODEL,
        training_file=training_file_id,
        validation_file=validation_file_id,
        method=method,
        suffix="bug-detection-oss",
        extra_body={"trainingType": TRAINING_TYPE},
    )
    state = {
        "equivalence": equivalence,
        "job_id": fine_tune_job.id,
        "training_file_id": training_file_id,
        "validation_file_id": validation_file_id,
        "submitted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    STATE_PATH.write_text(json.dumps(state, indent=2), encoding="utf-8")
    print(f"Submitted fresh job: {fine_tune_job.id}")

print(f"Job ID: {fine_tune_job.id}")
print(f"Initial status: {fine_tune_job.status}")


## 5. Monitor to terminal state


In [ ]:
completed_job = None
terminal_states = {"succeeded", "failed", "cancelled"}
seen_events = set()
deadline = time.monotonic() + 21_600
while True:
    completed_job = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    events = openai_client.fine_tuning.jobs.list_events(fine_tune_job.id, limit=100)
    for event in reversed(events.data):
        if event.id not in seen_events:
            seen_events.add(event.id)
            print(f"[{event.created_at}] {event.message}")
    print(f"Status: {completed_job.status}")
    if completed_job.status in terminal_states:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Job {fine_tune_job.id} exceeded 21600 seconds; rerun to resume.")
    time.sleep(60)

print(f"Terminal status: {completed_job.status}")

if completed_job.status != "succeeded":
    raise RuntimeError(f"Training ended as {completed_job.status}: {completed_job.error}")


## 6. Download service metrics and conclude

The conclusion is restricted to validation loss and validation token accuracy in service-produced result CSVs.


In [ ]:
run_dir = OUTPUT_DIR / fine_tune_job.id
run_dir.mkdir(parents=True, exist_ok=True)
result_paths = []
for index, file_id in enumerate(list(completed_job.result_files or []), start=1):
    payload = openai_client.files.content(
        file_id, extra_headers={"Accept-Encoding": "identity"}
    ).read()
    path = run_dir / f"result-{index}-{file_id}.csv"
    path.write_bytes(payload)
    result_paths.append(path)
    print(f"Downloaded {file_id} -> {path}")


def numeric_values(rows, names):
    values = []
    for row in rows:
        for name in names:
            raw = row.get(name)
            if raw not in (None, ""):
                try:
                    parsed = float(raw)
                    if not math.isfinite(parsed):
                        raise ValueError(f"Non-finite service metric {name}: {raw!r}")
                    values.append(parsed)
                    break
                except ValueError as exc:
                    raise ValueError(f"Invalid service metric {name}: {raw!r}") from exc
    return values


rows = []
columns = set()
for path in result_paths:
    with path.open("r", encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)
        columns.update(reader.fieldnames or [])
        rows.extend(reader)

validation_loss = numeric_values(
    rows, ("full_valid_loss", "full_validation_loss", "valid_loss", "validation_loss", "eval_loss")
)
validation_accuracy = numeric_values(
    rows,
    (
        "full_valid_mean_token_accuracy",
        "full_validation_mean_token_accuracy",
        "valid_mean_token_accuracy",
        "validation_mean_token_accuracy",
        "eval_mean_token_accuracy",
    ),
)


def curve(values):
    if not values:
        return None
    return {
        "points": values,
        "first": values[0],
        "final": values[-1],
        "minimum": min(values),
        "maximum": max(values),
    }


metrics = {
    "validation_loss": curve(validation_loss),
    "validation_token_accuracy": curve(validation_accuracy),
    "service_columns": sorted(columns),
}
if completed_job.status != "succeeded":
    conclusion = f"Training ended with status {completed_job.status}; no training-signal conclusion is supported."
elif metrics["validation_loss"] and metrics["validation_token_accuracy"]:
    conclusion = (
        f"Service validation loss changed from {validation_loss[0]:.6f} to {validation_loss[-1]:.6f}; "
        f"validation token accuracy changed from {validation_accuracy[0]:.6f} to "
        f"{validation_accuracy[-1]:.6f}. These are training signals only."
    )
elif metrics["validation_loss"]:
    conclusion = (
        f"Service validation loss changed from {validation_loss[0]:.6f} to {validation_loss[-1]:.6f}. "
        "No validation token-accuracy curve was emitted; this is training-loss evidence only."
    )
elif metrics["validation_token_accuracy"]:
    conclusion = (
        f"Service validation token accuracy changed from {validation_accuracy[0]:.6f} to "
        f"{validation_accuracy[-1]:.6f}. No validation-loss curve was emitted; this is training evidence only."
    )
else:
    raise RuntimeError("The service result files exposed no validation loss or validation token accuracy.")

summary = {
    "job_id": completed_job.id,
    "status": str(completed_job.status),
    "fine_tuned_model": completed_job.fine_tuned_model,
    "model": MODEL,
    "model_version": MODEL_VERSION,
    "training_type": TRAINING_TYPE,
    "hyperparameters": HYPERPARAMETERS,
    "datasets": dataset_evidence,
    "trained_tokens": completed_job.trained_tokens,
    "result_files": [str(path) for path in result_paths],
    "metrics": metrics,
    "conclusion": conclusion,
}
(run_dir / "run-summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
(OUTPUT_DIR / "run-summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print(json.dumps(summary, indent=2, default=str))
openai_client.close()
project_client.close()
credential.close()
